# LESSON 5.7: CT Reconstruction — Practical Applications
## Image Reconstruction from Projections

In this lesson:
- Fan-beam vs parallel-beam geometry
- Reconstruction artifacts and their causes
- Hounsfield units and CT windowing
- Complete CT reconstruction pipeline
- Biomedical applications of the Radon Transform

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, iradon
from skimage.data import shepp_logan_phantom
from skimage.transform import rescale
from skimage.draw import disk, ellipse

## 1. Fan-Beam vs Parallel-Beam Geometry

### Parallel-Beam Geometry
- Rays are **parallel** at each projection angle
- Source and detectors move together around the object
- Simpler mathematics (what we've studied so far)
- Used in early CT scanners (1st and 2nd generation)

### Fan-Beam Geometry
- Rays **diverge** from a single point source
- Single source, array of detectors
- Used in modern clinical CT scanners (3rd and 4th generation)
- Faster acquisition, but requires modified reconstruction

```
  Parallel-Beam:            Fan-Beam:
  
  | | | | | | |              * (point source)
  | | | | | | |             /|\
  v v v v v v v            / | \
  +-----------+           /  |  \
  |  Object   |         /   |   \
  +-----------+        / Object  \
  [Detectors ]        /    |      \
                     [  Detectors  ]
```

Fan-beam projections can be **rebinned** (rearranged) into equivalent parallel-beam projections.

In [ ]:
# Visualize parallel-beam vs fan-beam geometry
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Draw object
theta_obj = np.linspace(0, 2*np.pi, 100)

for ax in axes:
    ax.plot(2*np.cos(theta_obj), 2*np.sin(theta_obj), 'b-', linewidth=2)
    ax.fill(2*np.cos(theta_obj), 2*np.sin(theta_obj), alpha=0.1, color='blue')
    ax.set_xlim([-5, 5])
    ax.set_ylim([-5, 5])
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)

# Parallel-beam
for x in np.linspace(-2.5, 2.5, 8):
    axes[0].annotate('', xy=(x, -3.5), xytext=(x, 3.5),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
axes[0].plot([-3, 3], [3.5, 3.5], 'g-', linewidth=3, label='Source (at ∞)')
axes[0].plot([-3, 3], [-3.5, -3.5], 'm-', linewidth=3, label='Detector array')
axes[0].set_title('Parallel-Beam Geometry', fontsize=13)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Fan-beam
source = np.array([0, 4.5])
axes[1].plot(*source, 'r*', markersize=15, label='Point source')
detector_x = np.linspace(-3, 3, 10)
detector_y = -3.5 * np.ones_like(detector_x)
axes[1].plot([-3, 3], [-3.5, -3.5], 'm-', linewidth=3, label='Detector array')

for dx in detector_x:
    axes[1].plot([source[0], dx], [source[1], -3.5], 'r-', alpha=0.5, linewidth=1)
axes[1].set_title('Fan-Beam Geometry', fontsize=13)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')

plt.suptitle('CT Scanner Geometries', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Parallel-beam: All rays parallel — simpler math, slower acquisition.")
print("Fan-beam: Rays diverge from source — complex math, faster acquisition.")
print("Modern CT uses fan-beam (or cone-beam for 3D).")

## 2. Reconstruction Artifacts

Several types of artifacts can appear in CT reconstructions:

| Artifact | Cause | Appearance |
|----------|-------|------------|
| **Streak artifacts** | Too few projections | Star-like patterns |
| **Ring artifacts** | Miscalibrated detector elements | Concentric rings |
| **Metal artifacts** | High-density objects (implants) | Bright/dark streaks |
| **Motion artifacts** | Patient movement | Blurring/ghosting |
| **Partial volume** | Feature smaller than voxel | Blurred boundaries |
| **Limited-angle** | Incomplete angular coverage | Elongation/streaks |

In [ ]:
# Demonstrate common reconstruction artifacts
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta_full = np.linspace(0., 180., 360, endpoint=False)
sinogram_full = radon(phantom, theta=theta_full, circle=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Good reconstruction (reference)
recon_good = iradon(sinogram_full, theta=theta_full, filter_name='ramp', circle=True)
axes[0, 0].imshow(recon_good, cmap='gray')
axes[0, 0].set_title('Reference: 360 projections\n(good quality)', fontsize=11)
axes[0, 0].axis('off')

# 2. Streak artifacts (too few projections)
theta_sparse = np.linspace(0., 180., 15, endpoint=False)
sinogram_sparse = radon(phantom, theta=theta_sparse, circle=True)
recon_sparse = iradon(sinogram_sparse, theta=theta_sparse, filter_name='ramp', circle=True)
axes[0, 1].imshow(recon_sparse, cmap='gray')
axes[0, 1].set_title('Streak Artifacts\n(only 15 projections)', fontsize=11)
axes[0, 1].axis('off')

# 3. Limited-angle artifacts
theta_limited = np.linspace(0., 120., 120, endpoint=False)
sinogram_limited = radon(phantom, theta=theta_limited, circle=True)
recon_limited = iradon(sinogram_limited, theta=theta_limited, filter_name='ramp', circle=True)
axes[0, 2].imshow(recon_limited, cmap='gray')
axes[0, 2].set_title('Limited-Angle Artifacts\n(0°-120° only)', fontsize=11)
axes[0, 2].axis('off')

# 4. Ring artifacts (simulated by corrupting specific detector elements)
sinogram_ring = sinogram_full.copy()
# Add constant offset to certain detector rows (simulating bad pixels)
bad_detectors = [100, 150, 200, 250]
for d in bad_detectors:
    sinogram_ring[d, :] += 10
recon_ring = iradon(sinogram_ring, theta=theta_full, filter_name='ramp', circle=True)
axes[1, 0].imshow(recon_ring, cmap='gray')
axes[1, 0].set_title('Ring Artifacts\n(bad detector pixels)', fontsize=11)
axes[1, 0].axis('off')

# 5. Metal artifacts (high-density inclusion)
phantom_metal = phantom.copy()
size = phantom_metal.shape[0]
rr, cc = disk((size//2 + 20, size//2 - 10), 8)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
phantom_metal[rr[valid], cc[valid]] = 5.0  # very high density
sinogram_metal = radon(phantom_metal, theta=theta_full, circle=True)
recon_metal = iradon(sinogram_metal, theta=theta_full, filter_name='ramp', circle=True)
axes[1, 1].imshow(recon_metal, cmap='gray', vmin=-0.5, vmax=1.5)
axes[1, 1].set_title('Metal Artifacts\n(high-density implant)', fontsize=11)
axes[1, 1].axis('off')

# 6. Noise artifacts (low dose)
np.random.seed(42)
sinogram_noisy = sinogram_full + np.random.normal(0, 5.0, sinogram_full.shape)
recon_noisy = iradon(sinogram_noisy, theta=theta_full, filter_name='ramp', circle=True)
axes[1, 2].imshow(recon_noisy, cmap='gray')
axes[1, 2].set_title('Noise Artifacts\n(low dose, σ=5.0)', fontsize=11)
axes[1, 2].axis('off')

plt.suptitle('Common CT Reconstruction Artifacts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Hounsfield Units (HU)

CT images are displayed in **Hounsfield Units** (HU), which normalize attenuation values relative to water:

$$\boxed{HU = 1000 \times \frac{\mu - \mu_{\text{water}}}{\mu_{\text{water}}}}$$

| Material | HU Range |
|----------|----------|
| Air | -1000 |
| Lung | -500 to -900 |
| Fat | -100 to -50 |
| Water | 0 |
| Soft tissue | +20 to +80 |
| Blood | +30 to +45 |
| Bone (cancellous) | +100 to +300 |
| Bone (cortical) | +500 to +1900 |
| Metal | +2000+ |

In [ ]:
# Simulate CT image in Hounsfield Units
size = 256
ct_image = np.ones((size, size)) * (-1000)  # air background

# Body contour (soft tissue)
rr, cc = ellipse(128, 128, 100, 80)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
ct_image[rr[valid], cc[valid]] = 40  # soft tissue

# Lungs
rr, cc = ellipse(110, 90, 40, 25)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
ct_image[rr[valid], cc[valid]] = -700  # lung

rr, cc = ellipse(110, 166, 40, 25)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
ct_image[rr[valid], cc[valid]] = -700  # lung

# Spine (bone)
rr, cc = disk((170, 128), 15)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
ct_image[rr[valid], cc[valid]] = 800  # cortical bone

# Ribs
for angle in [30, 60, 120, 150]:
    rad = np.radians(angle)
    cx = int(128 + 75 * np.cos(rad))
    cy = int(128 - 75 * np.sin(rad))
    rr, cc = disk((cy, cx), 6)
    valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
    ct_image[rr[valid], cc[valid]] = 600  # bone

# Heart
rr, cc = ellipse(135, 140, 25, 20)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
ct_image[rr[valid], cc[valid]] = 35  # blood/heart

# Tumor (slightly different density)
rr, cc = disk((100, 155), 8)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
ct_image[rr[valid], cc[valid]] = 60  # tumor (denser than normal)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im = axes[0].imshow(ct_image, cmap='gray', vmin=-1000, vmax=1000)
axes[0].set_title('Simulated CT Image (Full HU Range)', fontsize=12)
plt.colorbar(im, ax=axes[0], label='Hounsfield Units (HU)')

# HU distribution
axes[1].hist(ct_image.ravel(), bins=100, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_title('HU Distribution', fontsize=12)
axes[1].set_xlabel('Hounsfield Units')
axes[1].set_ylabel('Pixel Count')
axes[1].axvline(x=0, color='cyan', linestyle='--', label='Water (0 HU)')
axes[1].axvline(x=-1000, color='white', linestyle='--', label='Air (-1000 HU)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CT Image in Hounsfield Units', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. CT Windowing

The full HU range (-1000 to +3000) is too wide for display. **Windowing** selects a subrange:

- **Window Width (WW)**: Range of HU values displayed
- **Window Level/Center (WL)**: Center of the displayed range

| Window Preset | WL | WW | Shows |
|--------------|----|----|-------|
| **Lung** | -600 | 1500 | Lung parenchyma |
| **Soft tissue** | 40 | 350 | Organs, muscles |
| **Bone** | 400 | 1800 | Skeletal structures |
| **Brain** | 40 | 80 | Brain tissue |
| **Liver** | 60 | 150 | Liver detail |

In [ ]:
# CT windowing demonstration
def apply_ct_window(image, window_level, window_width):
    """Apply CT windowing to map HU values to display range."""
    vmin = window_level - window_width / 2
    vmax = window_level + window_width / 2
    windowed = np.clip(image, vmin, vmax)
    windowed = (windowed - vmin) / (vmax - vmin)  # normalize to [0, 1]
    return windowed

# Window presets
windows = [
    ('Full Range', 0, 2000),
    ('Lung Window', -600, 1500),
    ('Soft Tissue', 40, 350),
    ('Bone Window', 400, 1800),
]

fig, axes = plt.subplots(1, len(windows), figsize=(18, 5))

for i, (name, wl, ww) in enumerate(windows):
    windowed = apply_ct_window(ct_image, wl, ww)
    axes[i].imshow(windowed, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f'{name}\nWL={wl}, WW={ww}', fontsize=11)
    axes[i].axis('off')

plt.suptitle('CT Windowing: Same Image, Different Window Settings',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Full Range: Everything visible but low contrast.")
print("Lung Window: Lung tissue clearly visible, soft tissue washed out.")
print("Soft Tissue: Organs and soft tissue visible, bones saturated.")
print("Bone Window: Skeletal structures visible, soft tissue dark.")

## 5. Complete CT Reconstruction Pipeline

Let's put everything together in a complete CT reconstruction pipeline.

In [ ]:
def ct_reconstruction_pipeline(phantom, n_projections=180, noise_std=0,
                               filter_name='shepp-logan', display_window=None):
    """
    Complete CT reconstruction pipeline.
    
    Parameters:
    -----------
    phantom : 2D array - the original image (ground truth)
    n_projections : int - number of projection angles
    noise_std : float - noise standard deviation
    filter_name : str - reconstruction filter name
    display_window : tuple (WL, WW) or None
    """
    # Step 1: Forward projection (simulate CT scan)
    theta = np.linspace(0., 180., n_projections, endpoint=False)
    sinogram = radon(phantom, theta=theta, circle=True)
    
    # Step 2: Add noise (simulate photon counting statistics)
    if noise_std > 0:
        sinogram_noisy = sinogram + np.random.normal(0, noise_std, sinogram.shape)
    else:
        sinogram_noisy = sinogram
    
    # Step 3: Filtered back projection (reconstruct)
    reconstruction = iradon(sinogram_noisy, theta=theta,
                           filter_name=filter_name, circle=True)
    
    # Step 4: Apply display window (if specified)
    if display_window is not None:
        wl, ww = display_window
        display = apply_ct_window(reconstruction, wl, ww)
    else:
        display = reconstruction
    
    return sinogram, sinogram_noisy, reconstruction, display

# Run the full pipeline
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

np.random.seed(42)
sinogram, sinogram_noisy, recon, display = ct_reconstruction_pipeline(
    phantom, n_projections=180, noise_std=1.0,
    filter_name='shepp-logan', display_window=None
)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Pipeline visualization
axes[0, 0].imshow(phantom, cmap='gray')
axes[0, 0].set_title('Step 0: Original Phantom', fontsize=11)

axes[0, 1].imshow(sinogram, cmap='hot', aspect='auto',
                  extent=[0, 180, sinogram.shape[0], 0])
axes[0, 1].set_title('Step 1: Clean Sinogram', fontsize=11)
axes[0, 1].set_xlabel('θ (degrees)')
axes[0, 1].set_ylabel('ρ')

axes[0, 2].imshow(sinogram_noisy, cmap='hot', aspect='auto',
                  extent=[0, 180, sinogram_noisy.shape[0], 0])
axes[0, 2].set_title('Step 2: Noisy Sinogram (σ=1.0)', fontsize=11)
axes[0, 2].set_xlabel('θ (degrees)')

axes[1, 0].imshow(recon, cmap='gray')
axes[1, 0].set_title('Step 3: FBP Reconstruction', fontsize=11)

# Error
min_dim = min(phantom.shape[0], recon.shape[0])
error = phantom[:min_dim, :min_dim] - recon[:min_dim, :min_dim]
rmse = np.sqrt(np.mean(error**2))

im_err = axes[1, 1].imshow(error, cmap='seismic', vmin=-0.1, vmax=0.1)
axes[1, 1].set_title(f'Reconstruction Error\nRMSE = {rmse:.4f}', fontsize=11)
plt.colorbar(im_err, ax=axes[1, 1])

# Profile comparison
center = min_dim // 2
axes[1, 2].plot(phantom[center, :min_dim], 'b-', linewidth=2, label='Original')
axes[1, 2].plot(recon[center, :min_dim], 'r--', linewidth=2, label='Reconstructed')
axes[1, 2].set_title('Line Profile (center row)', fontsize=11)
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Complete CT Reconstruction Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Parameter Study: Finding Optimal Settings

Let's explore how reconstruction quality depends on the parameters.

In [ ]:
# Parameter study: Projections × Noise × Filter
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

projection_counts = [30, 90, 180, 360]
noise_levels = [0, 1.0, 3.0]

np.random.seed(42)

fig, axes = plt.subplots(len(noise_levels), len(projection_counts),
                        figsize=(16, 12))

for i, noise_std in enumerate(noise_levels):
    for j, n_proj in enumerate(projection_counts):
        theta = np.linspace(0., 180., n_proj, endpoint=False)
        sinogram = radon(phantom, theta=theta, circle=True)
        
        if noise_std > 0:
            sinogram += np.random.normal(0, noise_std, sinogram.shape)
        
        # Use adaptive filter selection
        if noise_std == 0:
            filt = 'ramp'
        elif noise_std <= 1:
            filt = 'shepp-logan'
        else:
            filt = 'hamming'
        
        recon = iradon(sinogram, theta=theta, filter_name=filt, circle=True)
        
        min_dim = min(phantom.shape[0], recon.shape[0])
        error = phantom[:min_dim, :min_dim] - recon[:min_dim, :min_dim]
        rmse = np.sqrt(np.mean(error**2))
        
        axes[i, j].imshow(recon, cmap='gray')
        axes[i, j].set_title(f'RMSE={rmse:.3f} ({filt})', fontsize=9)
        axes[i, j].axis('off')
        
        if j == 0:
            axes[i, j].set_ylabel(f'σ = {noise_std}', fontsize=12)
        if i == 0:
            axes[i, j].set_title(f'{n_proj} proj\nRMSE={rmse:.3f} ({filt})', fontsize=9)

plt.suptitle('Parameter Study: Projections × Noise Level (with Adaptive Filter)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observations:")
print("• More projections → better quality (fewer streak artifacts)")
print("• More noise → need smoother filters (trade resolution for noise)")
print("• The optimal filter depends on both the noise level and the clinical task")

## 7. Biomedical Applications of the Radon Transform

The Radon Transform is fundamental to many medical imaging modalities:

| Modality | Uses Radon Transform |
|----------|---------------------|
| **CT (Computed Tomography)** | Direct application: X-ray projections → image reconstruction |
| **PET (Positron Emission Tomography)** | Reconstructs radiotracer distribution from coincidence data |
| **SPECT (Single-Photon Emission CT)** | Reconstructs gamma-ray emission from projections |
| **MRI (specific sequences)** | Radial k-space sampling uses Radon-like reconstruction |
| **Ultrasound (Synthetic Aperture)** | Some advanced techniques use projection-based methods |

### Beyond Medical Imaging:
- **Electron microscopy** (cryo-EM): Reconstructing 3D protein structures
- **Seismology**: Earth interior imaging from seismic wave data
- **Non-destructive testing**: Industrial CT for quality control
- **Astronomy**: Radio telescope aperture synthesis

In [ ]:
# Final comprehensive example: reconstructing a complex phantom
size = 256

# Create a detailed brain phantom
brain = np.ones((size, size)) * (-1000)  # air

# Skull
rr, cc = ellipse(128, 128, 110, 95)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 800  # skull bone

# Brain interior
rr, cc = ellipse(128, 128, 100, 85)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 40  # brain tissue (gray matter)

# White matter (slightly different density)
rr, cc = ellipse(128, 128, 70, 60)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 30  # white matter

# Ventricles (CSF)
rr, cc = ellipse(120, 115, 18, 8)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 5  # CSF

rr, cc = ellipse(120, 141, 18, 8)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 5  # CSF

# Third ventricle
rr, cc = ellipse(125, 128, 10, 3)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 5  # CSF

# Tumor
rr, cc = disk((100, 160), 15)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 65  # enhancing tumor

# Small calcification
rr, cc = disk((90, 155), 4)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
brain[rr[valid], cc[valid]] = 200  # calcification

# Normalize for Radon transform
brain_norm = (brain - brain.min()) / (brain.max() - brain.min())

# CT pipeline
np.random.seed(42)
theta = np.linspace(0., 180., 360, endpoint=False)
sinogram = radon(brain_norm, theta=theta, circle=True)
sinogram_noisy = sinogram + np.random.normal(0, 0.5, sinogram.shape)
recon = iradon(sinogram_noisy, theta=theta, filter_name='shepp-logan', circle=True)

# Rescale reconstruction back to HU
recon_hu = recon * (brain.max() - brain.min()) + brain.min()

# Display with different windows
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(brain, cmap='gray', vmin=-100, vmax=100)
axes[0, 0].set_title('Original (Brain Window)', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(sinogram_noisy, cmap='hot', aspect='auto',
                  extent=[0, 180, sinogram.shape[0], 0])
axes[0, 1].set_title('Sinogram (360 projections)', fontsize=11)
axes[0, 1].set_xlabel('θ')
axes[0, 1].set_ylabel('ρ')

axes[0, 2].imshow(recon_hu, cmap='gray', vmin=-100, vmax=100)
axes[0, 2].set_title('Reconstruction (Brain Window)', fontsize=11)
axes[0, 2].axis('off')

# Different windows on reconstruction
brain_w = apply_ct_window(recon_hu, 40, 80)
axes[1, 0].imshow(brain_w, cmap='gray')
axes[1, 0].set_title('Brain Window\n(WL=40, WW=80)', fontsize=11)
axes[1, 0].axis('off')

bone_w = apply_ct_window(recon_hu, 400, 1800)
axes[1, 1].imshow(bone_w, cmap='gray')
axes[1, 1].set_title('Bone Window\n(WL=400, WW=1800)', fontsize=11)
axes[1, 1].axis('off')

soft_w = apply_ct_window(recon_hu, 50, 200)
axes[1, 2].imshow(soft_w, cmap='gray')
axes[1, 2].set_title('Soft Tissue Window\n(WL=50, WW=200)', fontsize=11)
axes[1, 2].axis('off')

plt.suptitle('Complete Brain CT Simulation: Acquisition → Reconstruction → Windowing',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("This demonstrates the complete CT imaging chain:")
print("1. Object (brain phantom with tumor and calcification)")
print("2. Data acquisition (Radon Transform → sinogram + noise)")
print("3. Reconstruction (Filtered Back Projection with Shepp-Logan filter)")
print("4. Display (CT windowing for different tissue types)")

## Summary

What we learned:
1. **Fan-beam geometry** is used in modern CT scanners (faster than parallel-beam)
2. Common **reconstruction artifacts**: streak, ring, metal, motion, limited-angle
3. CT images are displayed in **Hounsfield Units** (HU): $HU = 1000 \times (\mu - \mu_{water}) / \mu_{water}$
4. **CT windowing** (WL, WW) selects which HU range to display — critical for diagnosis
5. The complete **CT reconstruction pipeline**: projection → noise → FBP → windowing
6. Optimal settings depend on **projection count**, **noise level**, and **clinical task**
7. The Radon Transform extends beyond CT to **PET, SPECT, MRI**, and many non-medical applications

### Module 5 Complete!
You now understand the full chain from the Radon Transform mathematics to practical CT reconstruction:
- Lesson 5.1: Radon Transform definition and geometry
- Lesson 5.2: Computing sinograms
- Lesson 5.3: Fourier Slice Theorem
- Lesson 5.4: Back projection and the 1/r blurring problem
- Lesson 5.5: Filtered Back Projection algorithm
- Lesson 5.6: Reconstruction filters and noise-resolution trade-off
- Lesson 5.7: CT reconstruction applications and clinical imaging